<a href="https://colab.research.google.com/github/sakuna47/real-time-multilingual-pii-detection/blob/Others/FYP_PP_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required dependencies
!pip install -q presidio-analyzer
!python -m spacy download en_core_web_lg -q

"""Pilot test: how well does out-of-the-box Microsoft Presidio detect
Sri Lankan NIC and phone numbers?  All data is synthetic.

Setup:
    pip install presidio-analyzer
    python -m spacy download en_core_web_lg
Run:
    python presidio_pilot.py
"""
import random
from collections import Counter
from presidio_analyzer import AnalyzerEngine

random.seed(42)  # fixed seed so the results are reproducible


def old_nic():  # 9 digits + V/X  (YY DDD SSSS + letter)
    return f"{random.randint(40, 99):02d}{random.randint(1, 866):03d}{random.randint(0, 9999):04d}{random.choice('VX')}"


def new_nic():  # 12 digits (YYYY DDD SSSSS)
    return f"{random.randint(1940, 2005)}{random.randint(1, 866):03d}{random.randint(0, 99999):05d}"


def phone():
    n = f"7{random.choice('01245678')}{random.randint(0, 9999999):07d}"
    return random.choice([f"+94{n}", f"+94 {n[:2]} {n[2:5]} {n[5:]}", f"0{n}", f"0{n[:2]} {n[2:5]} {n[5:]}", f"0094{n}"])


NIC_TEMPLATES = [
    "My NIC number is {v}.",
    "Please verify my identity, NIC {v}, for the loan application.",
    "Write a letter to the bank. My national identity card number is {v}.",
    "Fill this form for me: name Kasun Perera, NIC {v}, address Colombo 7.",
    "මගේ NIC එක {v} වේ.",
]
PHONE_TEMPLATES = [
    "Call me on {v} after 5pm.",
    "My mobile number is {v}.",
    "Draft a message saying customers can reach me at {v}.",
    "Contact Nimal on {v} for more information.",
    "මගේ phone number එක {v}.",
]

cases = []
for i in range(25):
    kind, val = ("NIC_OLD", old_nic()) if i < 13 else ("NIC_NEW", new_nic())
    cases.append((kind, val, NIC_TEMPLATES[i % 5].format(v=val)))
for i in range(25):
    val = phone()
    cases.append(("PHONE", val, PHONE_TEMPLATES[i % 5].format(v=val)))

analyzer = AnalyzerEngine()
found_any, found_right, labels = Counter(), Counter(), Counter()
total = Counter(k for k, _, _ in cases)

for kind, val, text in cases:
    start = text.index(val)
    end = start + len(val)
    hits = [r for r in analyzer.analyze(text=text, language="en") if r.start < end and r.end > start]
    if hits:
        found_any[kind] += 1
        for h in hits:
            labels[(kind, h.entity_type)] += 1
    # the whole value is covered AND given a sensible label
    right = {"PHONE": {"PHONE_NUMBER"}}.get(kind, set())  # Presidio has no Sri Lankan NIC label
    if any(h.entity_type in right and h.start <= start and h.end >= end for h in hits):
        found_right[kind] += 1

print(f"{'Type':10}{'Total':>7}{'Flagged as anything':>22}{'Correctly identified':>23}")
for k in ["NIC_OLD", "NIC_NEW", "PHONE"]:
    print(f"{k:10}{total[k]:>7}{found_any[k]:>22}{found_right[k]:>23}")
print("\nLabels Presidio gave to overlapping spans:")
for (k, lab), n in sorted(labels.items()):
    print(f"  {k:8} -> {lab:20} x{n}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Type        Total   Flagged as anything   Correctly identified
NIC_OLD        13                    13                      0
NIC_NEW        12                    12                      0
PHONE          25                    25                     22

Labels Presidio gave to overlapping spans:
  NIC_NEW  -> DATE_TIME            x5
  NIC_NEW  -> PERSON               x2
  NIC_NEW  -> US_BANK_NUMBER       x12
  NIC_NEW  -> US_DRIVER_LICENSE    x12
  NIC_OLD  -> DATE_TIME            x3
  NIC_OLD  -> PERSON               x2
  NIC_OLD  -> US_DRIVER_LICENSE    x13
  PHONE    -> DATE_TIME            x12
  PHONE    -> PHONE_NUMBER         x22
  PHONE    -> US_BANK_NUMBER       x17
  PHONE    -> US_DRIVER_LICENSE    x17
